# **Purpose**

**Task:** context + answer → question

This notebook fine-tunes a `meta-llama/Llama-3.2-3B-Instruct` model using 10000 sample set from the SQuAD V2 Dataset with QLoRA for MCQ Question and Answer Generation. It then saves the model into Google Drive for later use and evaluation.

## **Install the required modules**

In [1]:
!pip uninstall -y torchaudio torchvision

Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128


In [2]:
!pip install -qU \
  accelerate==1.14.0 \
  peft==0.20.0 \
  bitsandbytes==0.50.1 \
  transformers==5.15.1 \
  trl==1.10.0 \
  sentencepiece==0.2.2


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 13.9 MB/s eta 0:00:00



**Restart the session after all the modules are installed.**


##**Import the required Libraries**

In [3]:
# Standard Deep Learning and Hardware Framework
import torch
import transformers
import accelerate
import peft
import trl
import bitsandbytes as bnb
import sentencepiece as spm

# Hugging Face Datasets
from datasets import load_dataset

# Hugging Face Transformers & Pipelines
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline, TrainingArguments

# PEFT (Parameter-Efficient Fine-Tuning) / LoRA Tools
from peft import AutoPeftModelForCausalLM, LoraConfig, PeftModel, prepare_model_for_kbit_training, get_peft_model

# TRL (Transformer Reinforcement Learning) Trainers & Configs
from trl import SFTTrainer, SFTConfig, DPOConfig, DPOTrainer

# Utilities
from google.colab import drive
import os
import shutil

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("trl:", trl.__version__)
print("accelerate:", accelerate.__version__)
print("SentencePiece:", spm.__version__)
print("bitsandbytes:", bnb.__version__)


torch: 2.11.0+cu128
transformers: 5.15.1
peft: 0.20.0
trl: 1.10.0
accelerate: 1.14.0
SentencePiece: 0.2.2
bitsandbytes: 0.50.1


###**Mount Google Drive and set it up for storing the artifacts**

In [4]:
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [5]:
# Paths for intermediate checkpoints and final merged model
checkpoint_drive_path = '/content/drive/MyDrive/Fine_Tuned_Models/Llama-3.2/llama_3_2_3b_mcq_checkpoints'
adapter_drive_path = "/content/drive/MyDrive/Fine_Tuned_Models/Llama-3.2/Llama-3.2-3B-mcq-adapter"
merged_model_drive_path = "/content/drive/MyDrive/Fine_Tuned_Models/Llama-3.2/Llama-3.2-3B-mcq"

# Create folders if they do not exist
os.makedirs(checkpoint_drive_path, exist_ok=True)
os.makedirs(adapter_drive_path, exist_ok=True)
os.makedirs(merged_model_drive_path, exist_ok=True)

##**LLama 3.2 - 3 Biliion Chat Model**

###**Before Fine Tuning**

In [6]:
chat_model_name = "meta-llama/Llama-3.2-3B-Instruct"
#chat_model_name = "meta-llama/Llama-3.1-8B-Instruct"

In [ ]:
chat_tokenizer = AutoTokenizer.from_pretrained(chat_model_name)
chat_model = AutoModelForCausalLM.from_pretrained(chat_model_name)

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

In [ ]:
mcq_generator_chat = pipeline(
"text-generation",
model=chat_model,
tokenizer=chat_tokenizer,
return_full_text=False,
max_new_tokens=150,
do_sample=True,
temperature=0.7
)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


###**First Try**

In [ ]:
context = """
Photosynthesis is a biological process used by plants, algae, and certain bacteria to convert light energy into chemical energy stored in glucose. It occurs mainly in the chloroplasts of plant cells using chlorophyll pigments.
"""
target_answer = "chloroplasts"

messages = [
    {
        "role": "system",
        "content": "You are an expert educational assessment AI that generates a clear, high-quality multiple-choice question based strictly on given a context and target answer."
    },
    {
        "role": "user",
        "content": f"Context: {context}\nTarget Answer: {target_answer}\nGenerate a question from the given context where the target answer is the correct answer. Do not include phrases like 'According to the text' in the question and do not repeat the context in the question.\nThe output should be in the form\nQuestion:\nAnswer:"
    }
]

In [ ]:
output = mcq_generator_chat(messages)
output

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[{'generated_text': 'Where does photosynthesis mainly occur in plant cells? \n\nAnswer: chloroplasts'}]

In [ ]:
output[0]['generated_text']

'Where does photosynthesis mainly occur in plant cells? \n\nAnswer: chloroplasts'

###**Second Try**

In [ ]:
context = """
In operating systems, a deadlock is a situation where a set of processes are blocked because each process is holding a resource and waiting for another resource held by some other process. The Banker's algorithm is a classical resource allocation algorithm developed by Edsger Dijkstra to avoid deadlocks.
"""
target_answer = "Banker's algorithm"

messages = [
    {
        "role": "system",
        "content": "You are an expert educational assessment AI that generates a clear, high-quality multiple-choice question based strictly on given a context and target answer."
    },
    {
        "role": "user",
        "content": f"Context: {context}\nTarget Answer: {target_answer}\nGenerate a question from the given context where the target answer is the correct answer. Do not include phrases like 'According to the text' in the question and do not repeat the context in the question.\nThe output should be in the form\nQuestion:\nAnswer:"
    }
]

In [ ]:
output = mcq_generator_chat(messages)
output

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': "What algorithm was developed by Edsger Dijkstra to avoid deadlocks in operating systems?\n\nAnswer: Banker's algorithm"}]

In [ ]:
output[0]['generated_text']

"What algorithm was developed by Edsger Dijkstra to avoid deadlocks in operating systems?\n\nAnswer: Banker's algorithm"

**So as we see the chat model generates the multiple choice question from a context and target answer. But the questions generated do not elicit the answer given properly. So it needs to be trained to make the output optimal.**

##**Checking the Training Data**

###**Squad V2 Dataset**

In [7]:
train_dataset = load_dataset('rajpurkar/squad_v2', split='train').shuffle(seed=42).select(range(10000))

df = train_dataset.to_pandas()
df.head()

README.md:   0%|          | 0.00/8.92k [00:00<?, ?B/s]

squad_v2/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 16.4MB            

squad_v2/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

squad_v2/validation-00000-of-00001.parqu(…): reconstructing file:   0%|          |  0.00B / 1.35MB            

squad_v2/validation-00000-of-00001.parqu(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/130319 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11873 [00:00<?, ? examples/s]

,id,title,context,question,answers
0,56e0f3907aa994140058e80a,Canon_law,The Roman Catholic Church canon law also inclu...,What term characterizes the intersection of th...,"{'text': ['full union'], 'answer_start': [104]}"
1,571adcf932177014007e9f56,Athanasius_of_Alexandria,Alexandria was the most important trade center...,What was Alexandria known for?,"{'text': ['important trade center'], 'answer_s..."
2,57325b9fe99e3014001e670c,Jehovah%27s_Witnesses,Former members Heather and Gary Botting compar...,How do the leaders of the Jehovah's Witnesses ...,{'text': ['disparaging individual decision-mak...
3,5728d8be4b864d1900164f6b,Estonia,"Historically, the cuisine of Estonia has been ...",What are the most common foods in Estonia?,"{'text': ['black bread, pork, potatoes, and da..."
4,56f6f5e1711bf01900a44898,Classical_music,Many of the instruments used to perform mediev...,What was the medieval flute made from?,"{'text': ['wood'], 'answer_start': [126]}"


In [8]:
train_dataset

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 10000
})

###**Mapping Function**

In [9]:
def format_prompt(example, tokenizer):
    # Extract context and target answer safely (handling SQuAD v2 unanswerable questions)
    context = example["context"]
    answers = example.get("answers", {})

    if answers and len(answers.get("text", [])) > 0:
        target_answer = answers["text"][0]
    else:
        target_answer = "None"

    question = example.get("question", "")

    # Construct the conversational messages including system, user, and assistant turns
    messages = [
        {
            "role": "system",
            "content": "You are an expert educational assessment AI that generates a clear, high-quality question based strictly on a given context and target answer."
        },
        {
            "role": "user",
            "content": f"Context: {context}\nTarget Answer: {target_answer}\nGenerate a question from the given context where the target answer is the correct answer. Do not include phrases like 'According to the text' in the question and do not repeat the context in the question.\nThe output should be in the form\nQuestion:\nAnswer:"
        },
        {
            "role": "assistant",
            "content": f"Question: {question}\nAnswer: {target_answer}"
        }
    ]

    # Set a clean Llama-3 template without the date injection
    tokenizer.chat_template = (
        "{% set loop_messages = messages %}"
        "{% for message in loop_messages %}"
            "{{ '<|start_header_id|>' + message['role'] + '<|end_header_id|>\n\n' + message['content'] | trim + '<|eot_id|>' }}"
        "{% endfor %}"
        "{% if add_generation_prompt %}"
            "{{ '<|start_header_id|>assistant<|end_header_id|>\n\n' }}"
        "{% endif %}"
    )

    # Apply Llama's chat template to format the prompt for training
    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": formatted_text}

##**Model Quantization**

In [10]:
# 4-bit quantization configuration - Q in QLoRA
bnb_config = BitsAndBytesConfig(
  load_in_4bit=True,  # Use 4-bit precision model loading
  bnb_4bit_quant_type="nf4",  # Quantization type
  bnb_4bit_compute_dtype=torch.float16,  # Compute dtype
  bnb_4bit_use_double_quant=True,  # Apply nested quantization
)

In [11]:
# Load the model to train on the GPU
model = AutoModelForCausalLM.from_pretrained(
  chat_model_name,
  device_map="auto",
  dtype=torch.float16,
  # Leave this out for regular SFT
  quantization_config=bnb_config,
)

model.config.use_cache = False
model.config.pretraining_tp = 1

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [12]:
# Load LLaMA tokenizer
tokenizer = AutoTokenizer.from_pretrained(chat_model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [13]:
tokenized_train_dataset = train_dataset.map(lambda x: format_prompt(x,tokenizer))

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [14]:
tokenized_train_dataset

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers', 'text'],
    num_rows: 10000
})

In [15]:
print(tokenized_train_dataset["text"][2576])

<|start_header_id|>system<|end_header_id|>

You are an expert educational assessment AI that generates a clear, high-quality question based strictly on a given context and target answer.<|eot_id|><|start_header_id|>user<|end_header_id|>

Context: Tristan da Cunha /ˈtrɪstən də ˈkuːnjə/, colloquially Tristan, is both a remote group of volcanic islands in the south Atlantic Ocean and the main island of that group. It is the most remote inhabited archipelago in the world, lying 2,000 kilometres (1,200 mi) from the nearest inhabited land, Saint Helena, 2,400 kilometres (1,500 mi) from the nearest continental land, South Africa, and 3,360 kilometres (2,090 mi) from South America. The territory consists of the main island, also named Tristan da Cunha, which has a north–south length of 11.27 kilometres (7.00 mi) and has an area of 98 square kilometres (38 sq mi), along with the smaller, uninhabited Nightingale Islands and the wildlife reserves of Inaccessible and Gough Islands.
Target Answer: 

###**LoRA Configuration**

In [16]:
# Prepare LoRA Configuration
peft_config = LoraConfig(
  lora_alpha=32,  # LoRA Scaling
  lora_dropout=0.1,  # Dropout for LoRA Layers
  r=16,  # Rank
  bias="none",
  task_type="CAUSAL_LM",
  target_modules=  # Layers to target
  ["q_proj", "v_proj", "k_proj", "o_proj"]
  )

In [17]:
# Prepare model for training
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

###**Training Configuration**

In [18]:
training_arguments = SFTConfig(
  output_dir=checkpoint_drive_path,
  per_device_train_batch_size=4,
  gradient_accumulation_steps=8,
  optim="paged_adamw_8bit",
  learning_rate=2e-4,
  lr_scheduler_type="cosine",
  num_train_epochs=1,
  logging_steps=10,
  fp16=False,             # Disable fp16
  bf16=True,
  gradient_checkpointing=True,
  gradient_checkpointing_kwargs={"use_reentrant": False},
  save_strategy="steps",
  save_steps=10,
  save_total_limit=2,
  dataset_text_field="text",
  packing=True,
  max_length=512,
)

In [19]:
# Set supervised fine-tuning parameters
trainer = SFTTrainer(
  model=model,
  train_dataset=tokenized_train_dataset,
  args=training_arguments,
  processing_class=tokenizer,
)

Adding EOS to train dataset:   0%|          | 0/10000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/10000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/10000 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Step,Training Loss
10,2.555758
20,1.669843
30,1.557277
40,1.541483
50,1.535562
60,1.528069
70,1.529112


Step,Training Loss
10,2.555758
20,1.669843
30,1.557277
40,1.541483
50,1.535562
60,1.528069
70,1.529112
80,1.524315
90,1.534904
100,1.513755


###**WARNING! If Training fails run this cell. Else skip this cell.**

Resume Training from a particular Checkpoint. This looks into our Google Drive and loads the exact state from step XXX. Replace XXX with the correct checkpoint.

In [ ]:
checkpoint_dir = os.path.join(checkpoint_drive_path, "checkpoint-140")
trainer.train(resume_from_checkpoint=checkpoint_dir)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Step,Training Loss
150,1.494452
160,1.489159
170,1.525958
180,1.517848


###**Save the Adapter Files to Google Drive**

Save the final trained LoRA adapters to Google Collab folders locally first

In [ ]:

trainer.model.save_pretrained("Llama-3.2-3B-mcq")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Then Copy to Google Drive

In [ ]:
adapter_local_path = "/content/Llama-3.2-3B-mcq"

shutil.copytree(adapter_local_path, adapter_drive_path, dirs_exist_ok=True)

print(f"LoRA adapter files successfully copied to: {adapter_drive_path}")

LoRA adapter files successfully copied to: /content/drive/MyDrive/Fine_Tuned_Models/Tinyllama-1.1B-mcq-adapter


###**Merge Adapter**

In [ ]:
# Free up VRAM before loading the merge tool
del model
del trainer
torch.cuda.empty_cache()

In [ ]:
model = AutoPeftModelForCausalLM.from_pretrained(
    "Llama-3.2-3B-mcq",
    low_cpu_mem_usage=True,
    torch_dtype=torch.float16,
    device_map="auto",
)

# Merge LoRA and base model
merged_model = model.merge_and_unload()

###**Save and Download the Merged Model to Google Drive**

In [ ]:
# Save directly to your Google Drive
merged_model.save_pretrained(merged_model_drive_path)
tokenizer.save_pretrained(merged_model_drive_path)
print(f"Model and tokenizer safely saved to your Google Drive at: {merged_model_drive_path}")

Model and tokenizer safely saved to your Google Drive at: /content/drive/MyDrive/Fine_Tuned_Models/Tinyllama-1.1B-mcq


###**Using the Merged Model**

In [ ]:
# Use our predefined prompt template
context = """
In operating systems, a deadlock is a situation where a set of processes are blocked because each process is holding a resource and waiting for another resource held by some other process. The Banker's algorithm is a classical resource allocation algorithm developed by Edsger Dijkstra to avoid deadlocks.
"""
target_answer = "Banker's algorithm"

messages = [
    {
        "role": "system",
        "content": "You are an expert educational assessment AI that generates a clear, high-quality multiple-choice question based strictly on given a context and target answer."
    },
    {
        "role": "user",
        "content": f"Context: {context}\nTarget Answer: {target_answer}\nGenerate a question from the given context where the target answer is the correct answer. Do not include phrases like 'According to the text' in the question and do not repeat the context in the question.\nThe output should be in the form\nQuestion:\nAnswer:"
    }
]

# Run our instruction-tuned model
pipe = pipeline(
    task="text-generation",
    model=merged_model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=150,
    do_sample=True,
    temperature=0.7
    )
output = pipe(messages)
output

[{'generated_text': "Question: What algorithm is used to avoid deadlocks?\nAnswer: Banker's algorithm"}]

In [ ]:
output[0]['generated_text']

"Question: What algorithm is used to avoid deadlocks?\nAnswer: Banker's algorithm"